# Week 5: Segmentation & Morphology — Lab Practice

**Topics:** Thresholding, Mathematical Morphology, Connected Components

This notebook accompanies the Week 5 lecture slides. We will build a complete **segmentation pipeline** step by step: threshold an image to separate foreground from background, clean the result with morphological operations, and finally label and measure individual objects — all using a single image flowing through the entire pipeline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

%matplotlib inline

### Environment setup

This notebook works both **locally** and on **Google Colab**.
- **Local / Colab**: the rice image (used in the Final Challenge) is downloaded automatically on first run.

In [ ]:
import os, urllib.request

# Detect Google Colab
IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, "__IPYTHON__") else False

# Rice image for Final Challenge (downloaded at runtime — not stored in the repo)
RICE_URL = "https://raw.githubusercontent.com/Muzammillcoste/CV-Labs-CIS/master/imdata/rice.png"
RICE_PATH = "rice.png"

if not os.path.exists(RICE_PATH):
    print("Downloading rice.png ...")
    urllib.request.urlretrieve(RICE_URL, RICE_PATH)
    print("Done.")

print(f"Running on: {'Google Colab' if IN_COLAB else 'Local'}")

### Display helpers

Utility functions used throughout this notebook. This week adds `show_binary` for binary masks, `show_labeled` for color-coded connected component labels, and `overlay_contours` for displaying segmentation boundaries on the original image.

In [ ]:
def show_images(*imgs, titles=None, scale=4):
    """Display images in a single row.

    Grayscale (2-D) arrays automatically use a gray colormap.
    *titles* is an optional list of strings, one per image.
    """
    n = len(imgs)
    fig, axes = plt.subplots(1, n, figsize=(scale * n, scale))
    if n == 1:
        axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, imgs)):
        if img.ndim == 2:
            ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        else:
            ax.imshow(img)
        if titles:
            ax.set_title(titles[i])
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_binary(*masks, titles=None, scale=4):
    """Display binary masks (bool or 0/1) in a single row.

    White = foreground (True/1), Black = background (False/0).
    """
    n = len(masks)
    fig, axes = plt.subplots(1, n, figsize=(scale * n, scale))
    if n == 1:
        axes = [axes]
    for i, (ax, m) in enumerate(zip(axes, masks)):
        ax.imshow(m, cmap="gray", vmin=0, vmax=1)
        if titles:
            ax.set_title(titles[i])
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_with_hist(*imgs, titles=None, scale=4):
    """Display grayscale images (top row) with their histograms (bottom row)."""
    n = len(imgs)
    fig, axes = plt.subplots(2, n, figsize=(scale * n, scale * 1.5))
    if n == 1:
        axes = axes.reshape(2, 1)
    for i, img in enumerate(imgs):
        axes[0, i].imshow(img, cmap="gray", vmin=0, vmax=255)
        if titles:
            axes[0, i].set_title(titles[i])
        axes[0, i].axis("off")
        h, _ = np.histogram(img.ravel(), 256, [0, 256])
        axes[1, i].fill_between(np.arange(256), h, alpha=0.7)
        axes[1, i].set_xlim([0, 255])
        axes[1, i].set_xlabel("Pixel intensity")
    plt.tight_layout()
    plt.show()


def show_labeled(label_map, title="Labeled regions", scale=5):
    """Display a connected component label map with distinct colors.

    Background (label 0) is shown in light gray;
    each label gets a unique color via nipy_spectral colormap.
    """
    fig, ax = plt.subplots(figsize=(scale, scale))
    n_labels = label_map.max()
    if n_labels == 0:
        ax.imshow(label_map, cmap="gray")
    else:
        cmap = plt.cm.nipy_spectral
        display = np.zeros((*label_map.shape, 4))
        for i in range(1, n_labels + 1):
            display[label_map == i] = cmap(i / (n_labels + 1))
        display[label_map == 0] = [0.95, 0.95, 0.95, 1.0]
        ax.imshow(display)
    ax.set_title(f"{title} ({n_labels} objects)")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def overlay_contours(img, mask, color=(255, 0, 0)):
    """Overlay contours of a binary mask on a grayscale image.

    Returns an RGB uint8 image with colored contour outlines.
    """
    from scipy.ndimage import binary_dilation
    contour = binary_dilation(mask, iterations=1).astype(bool) ^ mask.astype(bool)
    if img.ndim == 2:
        rgb = np.stack([img, img, img], axis=2).copy()
    else:
        rgb = img.copy()
    rgb[contour] = color
    return rgb

---
## 1. Thresholding

Thresholding converts a grayscale image into a **binary** image by classifying each pixel as foreground or background based on its intensity:

$$B(x,y) = \begin{cases} 1 & \text{if } f(x,y) > T \\ 0 & \text{otherwise} \end{cases}$$

The key question: **how do we choose the threshold $T$?**

In [ ]:
from skimage.data import coins

img = coins()

print(f"Shape: {img.shape}, dtype: {img.dtype}")
print(f"Range: [{img.min()}, {img.max()}]")

show_with_hist(img, titles=["Coins image"])

### Global threshold

The simplest approach: pick a single threshold value $T$ for the entire image. Pixels above $T$ become foreground (white), below become background (black).

Let's try a few manual threshold values and see how sensitive the result is to our choice.

In [ ]:
thresholds = [80, 100, 120, 150]
masks = [img > t for t in thresholds]

show_binary(*masks, titles=[f"T = {t}" for t in thresholds])

### Otsu's method: automatic threshold selection

Manually picking $T$ is tedious and subjective. **Otsu's method** finds the optimal $T$ automatically by maximizing the **between-class variance** $\sigma_B^2$:

$$T^* = \arg\max_T \; \sigma_B^2(T) = \arg\max_T \; w_0(T) \cdot w_1(T) \cdot [\mu_0(T) - \mu_1(T)]^2$$

where:
- $w_0, w_1$: fraction of pixels in each class (background / foreground)
- $\mu_0, \mu_1$: mean intensity of each class

The idea: **the best threshold separates the two intensity groups as far apart as possible.**

In [ ]:
from skimage.filters import threshold_otsu

T_otsu = threshold_otsu(img)
binary_otsu = img > T_otsu

print(f"Otsu threshold: T = {T_otsu}")

# Visualize where Otsu places the threshold on the histogram
fig, ax = plt.subplots(figsize=(8, 3))
h, _ = np.histogram(img.ravel(), 256, [0, 256])
ax.fill_between(np.arange(256), h, alpha=0.7)
ax.axvline(T_otsu, color="red", linestyle="--", linewidth=2, label=f"Otsu T = {T_otsu}")
ax.set_xlabel("Pixel intensity")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

show_binary(binary_otsu, titles=[f"Otsu (T={T_otsu})"])

### When global thresholding fails: non-uniform illumination

A single threshold $T$ works well when the histogram is clearly **bimodal** (two distinct peaks). But if the illumination is **uneven** — one side of the image is brighter than the other — the histogram peaks overlap and no single $T$ can separate foreground from background everywhere.

Let's simulate this by applying a brightness gradient to the coins image.

In [ ]:
# Simulate uneven illumination
gradient = np.linspace(0.3, 1.0, img.shape[1])
coins_uneven = (img * gradient[np.newaxis, :]).astype(np.uint8)

# Otsu on the uneven image
T_uneven = threshold_otsu(coins_uneven)
binary_uneven = coins_uneven > T_uneven

show_with_hist(img, coins_uneven,
               titles=["Original (uniform)", "Uneven illumination"])
show_binary(binary_otsu, binary_uneven,
            titles=[f"Otsu on original (T={T_otsu})", f"Otsu on uneven (T={T_uneven}) — fails"])

### Adaptive thresholding

Adaptive thresholding computes a **different threshold for each pixel** based on the local neighborhood mean:

$$T(x,y) = \mu_W(x,y) - C$$

where $\mu_W(x,y)$ is the mean intensity in a window $W$ centered at $(x,y)$, and $C$ is a constant offset. A pixel is foreground if $f(x,y) > T(x,y)$.

This handles illumination gradients because each region gets its own threshold.

In [ ]:
from skimage.filters import threshold_local

# Adaptive threshold on the uneven image
block_size = 51  # must be odd — size of the local neighborhood
offset = 10      # C: subtracted from the local mean

T_local = threshold_local(coins_uneven, block_size, offset=offset)
binary_adaptive = coins_uneven > T_local

show_binary(binary_uneven, binary_adaptive,
            titles=[f"Global Otsu (T={T_uneven})", f"Adaptive (block={block_size}, C={offset})"])

### Pipeline step 1 complete

For the rest of this notebook, we continue the pipeline with the **original coins image** (uniform illumination) and its **Otsu threshold** result. Otsu works well here because the histogram is clearly bimodal.

In [ ]:
binary_coins = binary_otsu

print(f"Binary image — shape: {binary_coins.shape}, dtype: {binary_coins.dtype}")
print(f"Foreground pixels: {binary_coins.sum()} / {binary_coins.size} ({binary_coins.mean():.1%})")

show_binary(binary_coins, titles=["binary_coins (input to morphology)"])

### Exercises

**Exercise 1.1:** Implement Otsu's method **from scratch**.

Given the coins image (`img`):
1. Compute the histogram (256 bins, range 0-255)
2. For each candidate threshold $T$ (1 to 255), compute the between-class variance:
   $$\sigma_B^2(T) = w_0 \cdot w_1 \cdot (\mu_0 - \mu_1)^2$$
   where $w_0, w_1$ are the class weights (fractions of pixels below/above $T$) and $\mu_0, \mu_1$ are the class means
3. Find the $T$ that maximizes $\sigma_B^2$

Compare your result with `threshold_otsu(img)`.

*Hint:* Use `np.histogram(img.ravel(), 256, [0, 256])` to get bin counts. The cumulative sum of the histogram gives $w_0(T) = \sum_{i=0}^{T-1} h[i] / N$ at each $T$. Similarly, the cumulative sum of `hist * np.arange(256)` gives cumulative intensity sums for computing $\mu_0$ and $\mu_1$.

In [ ]:
# Input image
show_with_hist(img, titles=["img"])

# YOUR CODE HERE
# 1. hist, _ = np.histogram(...)
# 2. Loop over T = 1..255, compute sigma_B_sq for each T
# 3. my_otsu_T = T with maximum sigma_B_sq
# 4. my_binary = img > my_otsu_T


print(f"My Otsu threshold:     {my_otsu_T}")
print(f"skimage Otsu threshold: {threshold_otsu(img)}")

show_binary(my_binary, binary_otsu,
            titles=[f"My Otsu (T={my_otsu_T})", f"skimage Otsu (T={T_otsu})"])

**Exercise 1.2:** The adaptive threshold has two key parameters: `block_size` and `offset`. Explore what happens when you change the `offset` parameter.

Apply `threshold_local` to the **uneven coins image** (`coins_uneven`) with `block_size=51` and three different offsets: `0`, `10`, `20`. Display the three binary results side by side.

What role does the offset play? What happens when it is too small or too large?

*Hint:* A pixel is foreground when `coins_uneven > threshold_local(coins_uneven, 51, offset=c)`.

In [ ]:
# Input image
show_images(coins_uneven, titles=["coins_uneven"])

offsets = [0, 10, 20]

# YOUR CODE HERE
# For each offset in offsets, compute the adaptive binary result
# Store results in a list called results


show_binary(*results, titles=[f"offset = {c}" for c in offsets])

# What role does the offset play? What happens when it is too small or too large?
# (You may write your answer in Korean.)
#

---
## 2. Mathematical Morphology

The thresholded image looks reasonable, but zoom in and you will see:
1. **Noise**: small spurious foreground (white) or background (black) blobs
2. **Imperfections**: holes inside coins, rough boundaries

**Mathematical morphology** uses a small shape called a **structuring element** (SE) to probe and modify binary images. The four fundamental operations — **erosion**, **dilation**, **opening**, and **closing** — can clean up these issues.

In [ ]:
# Zoom in to see the issues
r1, r2, c1, c2 = 90, 200, 0, 150

show_binary(binary_coins, binary_coins[r1:r2, c1:c2],
            titles=["Full image", "Zoomed — notice noise and holes inside coins"])

### Structuring element

The structuring element is a small binary shape that slides over the image, probing the neighborhood at each pixel position. Common choices:
- **Cross** (4-connected): only horizontal and vertical neighbors
- **Square** (8-connected): includes diagonal neighbors

In [ ]:
# Cross: center + 4 direct neighbors (up, down, left, right)
se_cross = np.array([[0, 1, 0],
                     [1, 1, 1],
                     [0, 1, 0]], dtype=bool)

# Square: center + all 8 neighbors (including diagonals)
se_square = np.array([[1, 1, 1],
                      [1, 1, 1],
                      [1, 1, 1]], dtype=bool)

print(f"Cross (4-connected):\n{se_cross.astype(int)}")
print(f"\nSquare (8-connected):\n{se_square.astype(int)}")

### Erosion

Erosion **shrinks** foreground regions. A foreground pixel survives only if **all** pixels under the SE are foreground:

$$(A \ominus B)(x,y) = \min_{(i,j) \in B} A(x+i,\; y+j)$$

Effects: removes small noise, separates weakly connected objects, but shrinks all objects.

In [ ]:
from scipy.ndimage import binary_erosion

eroded = binary_erosion(binary_coins, structure=se_cross)

show_binary(binary_coins, eroded,
            titles=["Original binary", "Eroded (1 iteration, cross SE)"])

### Dilation

Dilation **expands** foreground regions. A background pixel becomes foreground if **any** pixel under the SE is foreground:

$$(A \oplus B)(x,y) = \max_{(i,j) \in B} A(x+i,\; y+j)$$

Effects: fills small holes, connects nearby objects, but enlarges all objects.

In [ ]:
from scipy.ndimage import binary_dilation

dilated = binary_dilation(binary_coins, structure=se_cross)

show_binary(binary_coins, dilated,
            titles=["Original binary", "Dilated (1 iteration, cross SE)"])

### Opening = Erosion followed by Dilation

Erosion alone removes noise but also shrinks objects. **Opening** fixes this: erode first (remove noise), then dilate (restore object size):

$$A \circ B = (A \ominus B) \oplus B$$

Opening removes **small foreground noise** without significantly shrinking large objects.

In [ ]:
from scipy.ndimage import binary_opening

opened = binary_opening(binary_coins, structure=se_cross, iterations=2)

show_binary(binary_coins, opened,
            titles=["After thresholding", "After opening (2 iterations, cross SE)"])

### Closing = Dilation followed by Erosion

**Closing** is the opposite of opening: dilate first (fill holes), then erode (restore boundaries):

$$A \bullet B = (A \oplus B) \ominus B$$

Closing fills **small background holes** inside foreground objects without enlarging them overall. Opening cleans the **outside**, closing cleans the **inside**.

In [ ]:
from scipy.ndimage import binary_closing

# Apply closing directly to the thresholded image
closed = binary_closing(binary_coins, structure=se_cross, iterations=2)

show_binary(binary_coins, closed,
            titles=["After thresholding", "After closing (2 iterations, cross SE)"])

<![CDATA[### Pipeline step 2 complete

In practice, we apply **opening** and **closing** together: opening removes foreground noise, then closing fills holes inside objects.

The morphologically cleaned binary image is now ready for connected component labeling.]]>

In [ ]:
clean_coins = binary_closing(opened, structure=se_cross, iterations=2)

print(f"Cleaned binary — foreground: {clean_coins.sum()} pixels ({clean_coins.mean():.1%})")

show_binary(binary_coins, opened, clean_coins,
            titles=["After thresholding", "After opening", "After opening + closing"])

<![CDATA[### Morphological gradient

The **morphological gradient** extracts object boundaries by computing the difference between dilation and erosion:

$$\text{gradient}(A) = (A \oplus B) - (A \ominus B)$$

This is a simple form of edge detection on binary images.]]>

In [ ]:
morph_gradient = binary_dilation(clean_coins, structure=se_cross).astype(int) - binary_erosion(clean_coins, structure=se_cross).astype(int)

show_binary(clean_coins, morph_gradient,
            titles=["Cleaned binary", "Morphological gradient (boundaries)"])

### Exercises

**Exercise 2.1:** Experiment with the **number of iterations** for opening on the thresholded coins image (`binary_coins`).

Apply `binary_opening` with iterations = 1, 3, and 5. Display the three results side by side.

At which iteration count do noise blobs disappear? What happens to the coins if you iterate too many times?

*Hint:* Use `binary_opening(binary_coins, iterations=n)` for each value.

In [ ]:
# Input
show_binary(binary_coins, titles=["binary_coins"])

iters = [1, 3, 5]

# YOUR CODE HERE
# Apply binary_opening to binary_coins with each iteration count
# Store results in a list called results


show_binary(*results, titles=[f"Opening (iter={n})" for n in iters])

# At which iteration count do noise blobs disappear?
# What happens to the coins if you iterate too many times?
# (You may write your answer in Korean.)
#

---
## 3. Connected Components

Now that we have a clean binary image, how do we **count** and **measure** the individual coins?

**Connected component labeling** assigns a unique integer label to each connected group of foreground pixels. Two pixels are "connected" if they share an edge or corner, depending on the chosen connectivity.

### 4-connectivity vs 8-connectivity

- **4-connected**: a pixel connects to its 4 direct neighbors (up, down, left, right)
- **8-connected**: a pixel connects to all 8 neighbors (including diagonals)

The choice affects how many objects are detected: 8-connectivity merges diagonally touching regions that 4-connectivity keeps separate.

In [ ]:
from scipy.ndimage import label

label_map, n_objects = label(clean_coins, structure=se_square)

print(f"Connected components found: {n_objects}")
print(f"Label map — shape: {label_map.shape}, range: [{label_map.min()}, {label_map.max()}]")

show_labeled(label_map, title="Connected component labeling (8-connected)")

### How labeling works: Flood Fill

Connected component labeling works by **flood fill**: starting from an unlabeled foreground pixel, it "floods" outward to all connected foreground neighbors, marking them with the same label. Then it moves to the next unlabeled pixel and repeats with a new label.

`scipy.ndimage.label` implements this efficiently. Each connected region gets a unique integer label (1, 2, 3, ...), while background stays 0.

### Region properties

Once we have labels, we can measure properties of each region:
- **Area**: number of pixels in the region
- **Centroid**: center of mass $(\bar{x}, \bar{y})$
- **Bounding box**: $(x_\text{min}, y_\text{min}, x_\text{max}, y_\text{max})$

In [ ]:
from skimage.measure import regionprops

regions = regionprops(label_map)

print(f"Number of regions: {len(regions)}\n")
print(f"{'Label':>6} {'Area':>8} {'Centroid':>20} {'BBox':>30}")
for r in regions:
    centroid = f"({r.centroid[0]:.1f}, {r.centroid[1]:.1f})"
    bbox = f"({r.bbox[0]}, {r.bbox[1]}, {r.bbox[2]}, {r.bbox[3]})"
    print(f"{r.label:>6} {r.area:>8} {centroid:>20} {bbox:>30}")

In [ ]:
# Visualize area distribution
areas = [r.area for r in regions]

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(areas, bins=30, edgecolor="black")
ax.set_xlabel("Area (pixels)")
ax.set_ylabel("Count")
ax.set_title(f"Distribution of region areas (n={len(areas)})")
plt.tight_layout()
plt.show()

print(f"Area statistics:")
print(f"  Mean:   {np.mean(areas):.1f} pixels")
print(f"  Median: {np.median(areas):.1f} pixels")
print(f"  Min:    {np.min(areas)} pixels")
print(f"  Max:    {np.max(areas)} pixels")

### Filtering by area

Small regions are likely **noise**, and very large regions may be **merged objects**. We can filter by area to keep only regions within a reasonable size range.

In [ ]:
min_area = 500

# Create a filtered label map: remove regions smaller than min_area
label_filtered = label_map.copy()
for r in regions:
    if r.area < min_area:
        label_filtered[label_map == r.label] = 0

# Re-label so labels are consecutive
label_filtered, n_filtered = label(label_filtered > 0, structure=se_square)

print(f"Before filtering: {n_objects} objects")
print(f"After filtering (area >= {min_area}): {n_filtered} objects")

show_labeled(label_map, title=f"All regions ({n_objects})")
show_labeled(label_filtered, title=f"After area filter ({n_filtered})")

### Complete pipeline

Let's see the full segmentation pipeline from start to finish:

1. **Grayscale** → 2. **Thresholding** (Otsu) → 3. **Morphology** (opening + closing) → 4. **Labeling** → 5. **Area filtering** → result

In [ ]:
# Full pipeline visualization
show_images(img, titles=["1. Grayscale input"])
show_binary(binary_coins, clean_coins,
            titles=[f"2. Otsu threshold (T={T_otsu})", "3. Morphological cleanup"])
show_labeled(label_filtered, title="4. Connected components")
show_images(overlay_contours(img, label_filtered > 0),
            titles=[f"5. Final result ({n_filtered} coins)"])

### Exercises

**Exercise 3.1:** Using the region properties (`regions` from `regionprops`), find and display:
1. The **largest** coin (by area)
2. The **smallest** coin (by area)

For each, create a binary mask that isolates just that one coin, and display both masks side by side. Also crop the original image to each coin's bounding box and display the crops.

*Hint:* Use `max(regions, key=lambda r: r.area)` to find the largest region. Create a mask with `label_map == r.label`. Use `r.bbox` (returns `min_row, min_col, max_row, max_col`) to crop.

In [ ]:
# Input
show_labeled(label_map, title=f"Labeled coins ({n_objects})")

# YOUR CODE HERE
# 1. Find the largest and smallest regions by area
#    largest  = max(regions, key=lambda r: r.area)
#    smallest = min(regions, key=lambda r: r.area)
# 2. Create binary masks: largest_mask  = (label_map == largest.label)
#                          smallest_mask = (label_map == smallest.label)
# 3. Crop original to each bounding box:
#    r = largest; largest_crop = img[r.bbox[0]:r.bbox[2], r.bbox[1]:r.bbox[3]]


print(f"Largest coin:  label={largest.label}, area={largest.area} pixels")
print(f"Smallest coin: label={smallest.label}, area={smallest.area} pixels")

show_binary(largest_mask, smallest_mask,
            titles=[f"Largest (area={largest.area})", f"Smallest (area={smallest.area})"])
show_images(largest_crop, smallest_crop,
            titles=["Largest (cropped)", "Smallest (cropped)"])

**Exercise 3.2:** Run the **entire pipeline** on the original coins image using the **Otsu binary** (`binary_otsu`) and compare the final coin count with the result from the adaptive demo.

Steps:
1. Start from `binary_otsu`
2. Apply morphological opening (2 iterations) + closing (2 iterations)
3. Label connected components (8-connectivity)
4. Filter by area (area >= 500)
5. Print the final coin count

*Hint:* Reuse the same morphology + labeling + filtering pattern from the demos above. The pipeline is identical — only the input binary changes.

In [ ]:
# Input
show_binary(binary_otsu, titles=["binary_otsu"])

# YOUR CODE HERE
# 1. binary_opening(binary_otsu, iterations=2)     → otsu_opened
# 2. binary_closing(otsu_opened, iterations=2)     → otsu_cleaned
# 3. label(otsu_cleaned, structure=se_square)       → otsu_labels, otsu_n
# 4. Filter by area >= 500, re-label                → otsu_filtered, otsu_n_filtered


print(f"Demo pipeline (adaptive):  {n_filtered} coins")
print(f"Otsu pipeline:             {otsu_n_filtered} coins")

show_labeled(label_filtered, title=f"Demo pipeline ({n_filtered})")
show_labeled(otsu_filtered, title=f"Otsu pipeline ({otsu_n_filtered})")

---
## Final Challenge

### Rice Grain Segmentation

You have practiced the full segmentation pipeline on the coins image. Now apply it to a **completely different image**: rice grains (`rice.png`).

The rice image has many small grains on a dark background with **non-uniform illumination** (brighter in the center, darker at the edges). Your task: **count the rice grains and measure their sizes.**

Follow the pipeline step by step:

1. **Load** the image and examine its histogram — is it bimodal?
2. **Threshold**: choose between Otsu and adaptive. Which works better given the illumination?
3. **Morphology**: apply opening + closing. Experiment with the number of iterations — rice grains are much smaller than coins, so the parameters will be different.
4. **Label**: use connected components to label individual grains.
5. **Filter**: remove very small regions (noise) by setting an area threshold.
6. **Report**: print the number of grains detected and their area statistics (mean, min, max).

Check results **visually** at each step and adjust parameters as needed. Use `overlay_contours` to display the final result on the original image.

*Hint:* Start by examining the histogram to decide between Otsu and adaptive. The rice image has illumination variation, so think about which method handles that better. For morphology, start with 1 iteration and increase if needed — rice grains are small so aggressive morphology may remove them.

In [ ]:
# Step 1: Load and examine
rice = np.array(Image.open(RICE_PATH))
print(f"Rice image — shape: {rice.shape}, dtype: {rice.dtype}")

show_with_hist(rice, titles=["Rice image"])

# YOUR CODE HERE
# Step 2: Threshold (choose Otsu or adaptive)
# Step 3: Morphology (opening + closing, tune iterations)
# Step 4: Label (connected components)
# Step 5: Filter by area
# Store final results in: rice_mask (bool), rice_labels (int), n_rice (int)


# Step 6: Report results
areas = [r.area for r in regionprops(rice_labels)]
print(f"\nRice grains detected: {n_rice}")
print(f"Area statistics:")
print(f"  Mean: {np.mean(areas):.1f} pixels")
print(f"  Min:  {np.min(areas)} pixels")
print(f"  Max:  {np.max(areas)} pixels")

# Overlay on original
rice_overlay = overlay_contours(rice, rice_mask)
show_images(rice_overlay, titles=[f"Result: {n_rice} rice grains"])
show_labeled(rice_labels, title=f"Labeled rice grains")